In [1]:
!pip install -q --upgrade Pillow timm albumentations kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 78.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.


In [2]:
import json, os

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "suyashjaiswal005", "key": "KGAT_c2842e6d8262e2a37714114438b4f390"}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d ciplab/real-and-fake-face-detection -p /content/data
!unzip -q /content/data/real-and-fake-face-detection.zip -d /content/data
!ls /content/data

Dataset URL: https://www.kaggle.com/datasets/ciplab/real-and-fake-face-detection
License(s): CC-BY-NC-SA-4.0
100% 431M/431M [00:05<00:00, 83.8MB/s]

real_and_fake_face	      real-and-fake-face-detection.zip
real_and_fake_face_detection


In [4]:
DATA_DIR = "/content/data/real_and_fake_face"
print(os.listdir(DATA_DIR))
# Should show: ['training_real', 'training_fake']

['training_real', 'training_fake']


In [5]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
EPOCHS     = 5
LR         = 1e-4
IMG_SIZE   = 224

print(f"Device: {DEVICE}")  # must say cuda
print(f"Torch: {torch.__version__}")

Device: cuda
Torch: 2.10.0+cu128


In [9]:
from sklearn.model_selection import train_test_split

class FaceDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = np.array(Image.open(path).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label

# Collect all samples
all_samples = []
for fname in os.listdir(os.path.join(DATA_DIR, "training_real")):
    if fname.lower().endswith((".jpg", ".png", ".jpeg")):
        all_samples.append((os.path.join(DATA_DIR, "training_real", fname), 0))  # real=0

for fname in os.listdir(os.path.join(DATA_DIR, "training_fake")):
    if fname.lower().endswith((".jpg", ".png", ".jpeg")):
        all_samples.append((os.path.join(DATA_DIR, "training_fake", fname), 1))  # fake=1

print(f"Total samples: {len(all_samples)}")

# Split into train/val (90/10)
train_samples, val_samples = train_test_split(all_samples, test_size=0.1, random_state=42)
print(f"Train: {len(train_samples)} | Val: {len(val_samples)}")

train_dataset = FaceDataset(train_samples, transform=train_transform)
val_dataset   = FaceDataset(val_samples,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Labels → real=0, fake=1")

Total samples: 2041
Train: 1836 | Val: 205
Labels → real=0, fake=1


In [10]:
model = timm.create_model("efficientnet_b4", pretrained=True, num_classes=2)
model = model.to(DEVICE)
print("Model ready ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Model ready ✅


In [11]:
from google.colab import drive
drive.mount('/content/drive')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
best_val_acc = 0

for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")

    # Train
    model.train()
    total_loss, correct = 0, 0
    for imgs, labels in tqdm(train_loader, desc="Training"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += (out.argmax(1) == labels).sum().item()
    print(f"Train Loss: {total_loss/len(train_loader):.4f} | Acc: {correct/len(train_dataset)*100:.2f}%")

    # Validate
    model.eval()
    correct = 0
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Validating"):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            correct += (out.argmax(1) == labels).sum().item()
    val_acc = correct / len(val_dataset)
    print(f"Val Acc: {val_acc*100:.2f}%")

    scheduler.step()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/drive/MyDrive/deepfake_model_ff.pth")
        print("✅ Best model saved to Drive!")

print(f"\n🏁 Done! Best Val Acc: {best_val_acc*100:.2f}%")

Mounted at /content/drive

--- Epoch 1/5 ---


Training: 100%|██████████| 58/58 [00:28<00:00,  2.02it/s]


Train Loss: 1.1940 | Acc: 53.65%


Validating: 100%|██████████| 7/7 [00:01<00:00,  4.01it/s]


Val Acc: 52.68%
✅ Best model saved to Drive!

--- Epoch 2/5 ---


...

Val Acc: 56.10%
✅ Best model saved to Drive!

--- Epoch 5/5 ---


Training: 100%|██████████| 58/58 [00:30<00:00,  1.92it/s]


Train Loss: 0.5813 | Acc: 71.73%


Validating: 100%|██████████| 7/7 [00:02<00:00,  2.56it/s]

Val Acc: 52.68%

🏁 Done! Best Val Acc: 56.10%


In [18]:
import torch
import timm
from torchvision import transforms
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

model = timm.create_model("efficientnet_b4", pretrained=False, num_classes=2)
model.load_state_dict(torch.load("/content/drive/MyDrive/deepfake_model.pth", map_location=DEVICE))
model.to(DEVICE)
model.eval()

# Upload a known fake image and test
from google.colab import files
uploaded = files.upload()

for fname in uploaded:
    img = Image.open(fname).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = model(tensor)
        probs = torch.softmax(out, dim=1)[0]
    print(f"Class 0 (fake): {probs[0].item()*100:.2f}%")
    print(f"Class 1 (real): {probs[1].item()*100:.2f}%")
    print(f"Predicted: {'REAL' if probs[1] > probs[0] else 'FAKE'}")

Saving Screenshot 2026-04-15 194356.png to Screenshot 2026-04-15 194356.png
Class 0 (fake): 99.59%
Class 1 (real): 0.41%
Predicted: FAKE
